In [1]:
# Cell 1 — Libraries needed
%pip install -q pypdf python-docx reportlab

In [11]:
# Cell 2 —
import subprocess
from pathlib import Path

BASE_URL = "https://ai-learning-repo.github.io/resources/rag"  # Replace with your actual folder/raw GitHub URL

md_files = [
    "trip_planner_itinerary.md",
    "trip_planner_policies_pricing.md",
    "trip_planner_lodges_emergencies.md",
]

for filename in md_files:
    file_path = Path(filename)

    # Check if file exists and is non-empty
    if file_path.exists() and file_path.stat().st_size > 0:
        print(f"[OK] Found in Colab: {file_path.resolve()}")
        continue

    url = f"{BASE_URL}/{filename}"
    print(f"[MISSING] Downloading {filename} from {url} ...")

    try:
        subprocess.run(["wget", "-q", "-O", str(file_path), url], check=True)
        if file_path.stat().st_size == 0:
            raise ValueError("Downloaded file is empty.")
        print(f"[DOWNLOADED] {filename}")
    except Exception as e:
        # Remove partial/0-byte file so the next run doesn't treat it as valid
        if file_path.exists():
            file_path.unlink()
        print(f"[ERROR] Failed to download {filename}: {e}")

[OK] Found in Colab: /content/trip_planner_itinerary.md
[OK] Found in Colab: /content/trip_planner_policies_pricing.md
[OK] Found in Colab: /content/trip_planner_lodges_emergencies.md


In [12]:
# Cell 3 — Convert existing .md files into .docx and .pdf (plus verify extraction)
import html
import re
from pathlib import Path

import docx
from pypdf import PdfReader
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer


def md_inline_to_reportlab(text: str) -> str:
    """Escape XML chars and convert **bold** markdown to <b>bold</b> for ReportLab."""
    escaped = html.escape(text)
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", escaped)


def md_to_docx(md_path: Path, docx_path: Path) -> None:
    """Convert a Markdown file into a structured Word (.docx) document."""
    doc = docx.Document()
    lines = md_path.read_text(encoding="utf-8").splitlines()

    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            continue

        # Strip **bold** markers for plain clean Word text (or keep readable)
        clean_line = re.sub(r"\*\*(.+?)\*\*", r"\1", line)

        if line.startswith("### "):
            doc.add_heading(clean_line[4:], level=3)
        elif line.startswith("## "):
            doc.add_heading(clean_line[3:], level=2)
        elif line.startswith("# "):
            doc.add_heading(clean_line[2:], level=1)
        elif line.startswith("- "):
            doc.add_paragraph(clean_line[2:], style="List Bullet")
        else:
            doc.add_paragraph(clean_line)

    doc.save(docx_path)


def md_to_pdf(md_path: Path, pdf_path: Path) -> None:
    """Convert a Markdown file into a multi-page, text-wrapped PDF using ReportLab Platypus."""
    styles = getSampleStyleSheet()
    body_style = styles["Normal"]
    body_style.fontSize = 10.5
    body_style.leading = 14

    bullet_style = ParagraphStyle(
        "BulletCustom",
        parent=body_style,
        leftIndent=18,
        firstLineIndent=-10,
    )

    story = []
    lines = md_path.read_text(encoding="utf-8").splitlines()

    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            story.append(Spacer(1, 6))
            continue

        if line.startswith("### "):
            story.append(Paragraph(md_inline_to_reportlab(line[4:]), styles["Heading3"]))
            story.append(Spacer(1, 4))
        elif line.startswith("## "):
            story.append(Paragraph(md_inline_to_reportlab(line[3:]), styles["Heading2"]))
            story.append(Spacer(1, 6))
        elif line.startswith("# "):
            story.append(Paragraph(md_inline_to_reportlab(line[2:]), styles["Heading1"]))
            story.append(Spacer(1, 8))
        elif line.startswith("- "):
            story.append(Paragraph("• " + md_inline_to_reportlab(line[2:]), bullet_style))
            story.append(Spacer(1, 3))
        else:
            story.append(Paragraph(md_inline_to_reportlab(line), body_style))
            story.append(Spacer(1, 6))

    pdf_doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
    pdf_doc.build(story)


# Save the 3 markdown files in your working directory first, then run this loop:
md_files = [
    "trip_planner_itinerary.md",
    "trip_planner_policies_pricing.md",
    "trip_planner_lodges_emergencies.md",
]

for filename in md_files:
    md_path = Path(filename)
    if not md_path.exists():
        print(f"Skipping {filename} (not found in current folder)")
        continue

    docx_path = md_path.with_suffix(".docx")
    pdf_path = md_path.with_suffix(".pdf")

    md_to_docx(md_path, docx_path)
    md_to_pdf(md_path, pdf_path)

    # Quick sanity check: verify text extraction & length for chunking
    md_chars = len(md_path.read_text(encoding="utf-8"))
    docx_obj = docx.Document(docx_path)
    docx_chars = sum(len(p.text) for p in docx_obj.paragraphs)
    pdf_reader = PdfReader(pdf_path)
    pdf_chars = sum(len(page.extract_text() or "") for page in pdf_reader.pages)

    print(
        f"Converted {md_path.stem}: "
        f"MD={md_chars} chars | DOCX={docx_chars} chars | "
        f"PDF={pdf_chars} chars ({len(pdf_reader.pages)} pages)"
    )

Converted trip_planner_itinerary: MD=5492 chars | DOCX=5330 chars | PDF=5388 chars (3 pages)
Converted trip_planner_policies_pricing: MD=4842 chars | DOCX=4657 chars | PDF=4714 chars (2 pages)
Converted trip_planner_lodges_emergencies: MD=5021 chars | DOCX=4828 chars | PDF=4876 chars (3 pages)
